# 03 — Model Comparison

Compare the 4 trained models and explain why the champion wins.

Sections:
1. Load MLflow experiment results
2. PR-AUC comparison across models
3. Precision-Recall curves — the right way to compare fraud models
4. Calibration plots — are probability outputs reliable?
5. Threshold analysis — what FP/FN trade-off do we accept?
6. Champion model explanation

**LinkedIn post angle:** *'I trained 4 models on fraud data. Here's why LightGBM won — and what the logistic regression baseline told me that the winner couldn\'t.'*

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import precision_recall_curve
from pathlib import Path
import sys

sys.path.append('..')
mlflow.set_tracking_uri('http://localhost:5001')

## 1. Load MLflow results

In [ ]:
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name('fintech-fraud-detection')

if experiment:
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=['metrics.val_pr_auc DESC']
    )
    results = pd.DataFrame([
        {
            'model': r.data.params.get('model_type', r.info.run_name),
            'val_pr_auc': r.data.metrics.get('val_pr_auc'),
            'test_pr_auc': r.data.metrics.get('test_pr_auc'),
            'val_roc_auc': r.data.metrics.get('val_roc_auc'),
            'val_f1': r.data.metrics.get('val_f1'),
        }
        for r in runs
    ])
    print(results.to_string())
else:
    print('No MLflow experiment found. Run `make train` first.')

## 2. PR-AUC comparison

In [ ]:
if 'results' in dir() and not results.empty:
    fig = go.Figure()
    x = results['model']
    fig.add_bar(x=x, y=results['val_pr_auc'], name='Validation PR-AUC')
    fig.add_bar(x=x, y=results['test_pr_auc'], name='Test PR-AUC')
    fig.update_layout(
        title='PR-AUC by Model',
        barmode='group',
        yaxis_title='PR-AUC',
        yaxis_range=[0, 1],
    )
    # Add baseline: a random classifier has PR-AUC ≈ fraud_rate ≈ 0.035
    fig.add_hline(y=0.035, line_dash='dash', line_color='gray',
                  annotation_text='Random baseline (3.5% fraud rate)')
    fig.show()

## 3. Precision-Recall curves — operating threshold analysis

In [ ]:
# Load champion model and test set
import joblib
from src.features.build_features import build_features

try:
    model = joblib.load('../models/champion/model.pkl')
    encoders = joblib.load('../models/champion/encoders.pkl')

    df = pd.read_parquet('../data/processed/merged.parquet')
    df = df.sort_values('TransactionDT').reset_index(drop=True)
    test_df = df.iloc[int(len(df)*0.85):]

    X_test, y_test, _ = build_features(test_df.copy(), fit=False, encoders=encoders)
    X_test = X_test.fillna(-999)
    y_prob = model.predict_proba(X_test)[:, 1]

    precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=recall, y=precision, mode='lines', name='Champion model'
    ))
    fig.add_vline(x=0.80, line_dash='dash', line_color='orange',
                  annotation_text='80% recall SLA')
    fig.update_layout(
        title='Precision-Recall Curve — Champion Model',
        xaxis_title='Recall (fraction of fraud caught)',
        yaxis_title='Precision (fraction of flags that are real fraud)',
    )
    fig.show()

    # At 80% recall, what precision can we achieve?
    mask = recall[:-1] >= 0.80
    if mask.any():
        prec_at_80 = precision[:-1][mask].max()
        thresh_at_80 = thresholds[mask].min()
        print(f'At 80% recall: precision = {prec_at_80:.2%}, threshold = {thresh_at_80:.3f}')
        print(f'Interpretation: to catch 80% of fraud, {1-prec_at_80:.1%} of our flags will be false alarms.')
except FileNotFoundError:
    print('Run `make train` first to generate champion model.')

## 4. Threshold analysis — business decision

The threshold is a business parameter. This cell helps you choose it.

- **Lower threshold** = catch more fraud, but more false positives (blocked legitimate transactions)
- **Higher threshold** = fewer false positives, but more missed fraud

For a BNPL company like Tabby: missing fraud is expensive (credit loss).
For a card network like Network International: too many false positives hurt customer experience.
The right threshold depends on the business context.

In [ ]:
if 'thresholds' in dir():
    threshold_df = pd.DataFrame({
        'threshold': thresholds,
        'precision': precision[:-1],
        'recall': recall[:-1],
        'f1': 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-8),
    })

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=threshold_df['threshold'], y=threshold_df['precision'],
                             name='Precision', mode='lines'))
    fig.add_trace(go.Scatter(x=threshold_df['threshold'], y=threshold_df['recall'],
                             name='Recall', mode='lines'))
    fig.add_trace(go.Scatter(x=threshold_df['threshold'], y=threshold_df['f1'],
                             name='F1', mode='lines'))
    fig.add_vline(x=0.3, line_dash='dash', line_color='green', annotation_text='APPROVE (0.3)')
    fig.add_vline(x=0.7, line_dash='dash', line_color='red', annotation_text='DECLINE (0.7)')
    fig.update_layout(
        title='Precision / Recall / F1 vs Threshold',
        xaxis_title='Decision Threshold',
        yaxis_title='Score',
        yaxis_range=[0, 1],
    )
    fig.show()